# Previsione

Carica un modello addestrato, verifica che i dati siano aggiornati, e produce la
previsione a tre giorni sull'intero dominio.

**Che tipo di previsione e'.** ERA5 pubblica con circa sei giorni di ritardo, quindi
l'ultima finestra disponibile non finisce oggi. Cio' che si ottiene e' una previsione
su giorni gia' trascorsi, confrontabile con l'osservato: un limite della sorgente, non
del modello, ed e' anche cio' che permette di verificarne l'onesta'.

In [ ]:
import sys
from pathlib import Path

# Il notebook puo' essere aperto dalla cartella `notebooks/`: senza questo, l'import
# di `dwf` fallisce a seconda di dove e' stato avviato Jupyter.
RADICE = Path.cwd()
if not (RADICE / "src").exists():
    RADICE = RADICE.parent
sys.path.insert(0, str(RADICE / "src"))

from dwf.config import Config

config = Config.load(RADICE / "configs" / "default.yaml", project_root=RADICE)
print(f"periodo   : {config.time.start} .. {config.time.end}")
print(f"dominio   : {config.region.n_lat} x {config.region.n_lon}")
print(f"finestre  : {config.windows.input_slots} slot in ingresso -> "
      f"{config.windows.output_slots} previsti")
print(f"dati      : {config.paths.data_root}")

## 1. I dati sono aggiornati?

Se mancano mesi, la finestra piu' recente utilizzabile e' vecchia. Le celle seguenti
mostrano che cosa c'e' e che cosa manca; il download e l'ingestione si lanciano dai
rispettivi script.

In [ ]:
from dwf.data.ingest import available_months

attesi = config.time.months()
presenti = available_months(config)
mancanti = [mese for mese in attesi if mese not in presenti]

print(f"mesi attesi   : {len(attesi)}")
print(f"GRIB presenti : {len(presenti)}")
if mancanti:
    print(f"mancanti      : {len(mancanti)}, dal {mancanti[0][0]}-{mancanti[0][1]:02d} "
          f"al {mancanti[-1][0]}-{mancanti[-1][1]:02d}")
    print("\nPer scaricarli:")
    print(f"  python scripts/download_era5.py --from-month {mancanti[0][0]}-{mancanti[0][1]:02d}")
    print("  python scripts/ingest_era5.py")
else:
    print("nessun mese mancante")

In [ ]:
import numpy as np

from dwf.tables import SLOTS, read_table

catalogo = read_table(SLOTS, config.tables_dir).sort("slot_index")
utilizzabili = catalogo.get_column("usable").to_numpy()
ultimo = catalogo.filter("usable").tail(1)
print(f"slot utilizzabili: {int(utilizzabili.sum())} su {len(utilizzabili)}")
print(f"ultimo istante disponibile: {ultimo.get_column('valid_time').item()}")

## 2. Il modello

Il checkpoint contiene i pesi e il numero di canali attesi; le statistiche di
normalizzazione stanno accanto. Se la configurazione e' cambiata dopo l'addestramento,
il caricamento fallisce invece di produrre previsioni senza senso.

In [ ]:
from dwf.train import load_checkpoint

FOLD = 0
rete, stats, input_layout, output_layout = load_checkpoint(config, FOLD)
print(f"canali attesi: {input_layout.n_channels}")
print(f"parametri    : {rete.n_parameters:,}")
print(f"normalizzazione calcolata su: {stats.computed_on_split}")

## 3. La previsione

Un solo passaggio della rete produce tutti e nove gli slot previsti: non c'e'
ricorsione, quindi non c'e' accumulo di errore da un passo al successivo.

In [ ]:
from dwf.data.dataset import build_reader
from dwf.predict import latest_usable_start, predict_window, summarize

lettore = build_reader(config, input_layout)
inizio = latest_usable_start(config, utilizzabili)

previsione = predict_window(
    config, rete, input_layout, output_layout, stats, lettore, inizio
)
print(f"ultimo istante osservato: {previsione.init_time}")
summarize(previsione)

## 4. Le mappe

Tre grandezze per ciascuna scadenza: temperatura prevista, probabilita' di
precipitazione e probabilita' che sia neve. L'incertezza sulla temperatura e' una
previsione a sua volta, e va guardata insieme alla media.

In [ ]:
import matplotlib.pyplot as plt

estensione = [config.region.west, config.region.east, config.region.south, config.region.north]

def mappa(asse, campo, titolo, cmap, vmin=None, vmax=None):
    immagine = asse.imshow(campo, extent=estensione, origin="upper", cmap=cmap,
                           vmin=vmin, vmax=vmax, aspect="auto")
    asse.set_title(titolo, fontsize=10)
    plt.colorbar(immagine, ax=asse, fraction=0.03)

SCADENZE = [0, 4, 8]  # primo giorno, secondo, terzo
figura, assi = plt.subplots(len(SCADENZE), 3, figsize=(15, 4 * len(SCADENZE)))
for riga, scadenza in enumerate(SCADENZE):
    istante = previsione.valid_times[scadenza]
    mappa(assi[riga, 0], previsione.t2m_mean[scadenza] - 273.15,
          f"Temperatura [C] - {istante:%d/%m %H UTC}", "RdBu_r")
    mappa(assi[riga, 1], previsione.precip_probability[scadenza],
          f"Probabilita' di pioggia - {istante:%d/%m %H UTC}", "Blues", 0, 1)
    mappa(assi[riga, 2], previsione.snow_probability[scadenza],
          f"Probabilita' di neve - {istante:%d/%m %H UTC}", "PuBu", 0, 1)
plt.tight_layout(); plt.show()

In [ ]:
# L'incertezza dichiarata dal modello: dove e' alta, la previsione va presa con cautela.
figura, assi = plt.subplots(1, 3, figsize=(15, 4))
for colonna, scadenza in enumerate(SCADENZE):
    mappa(assi[colonna], previsione.t2m_std[scadenza],
          f"Incertezza [K] - +{scadenza} slot", "magma")
plt.tight_layout(); plt.show()

## 5. Previsione in un punto

Utile per leggere il risultato come lo leggerebbe una persona: che tempo fa in un
luogo, nei prossimi tre giorni.

In [ ]:
LATITUDINE, LONGITUDINE = 45.07, 7.69  # Torino

riga = int(np.abs(previsione.latitudes - LATITUDINE).argmin())
colonna = int(np.abs(previsione.longitudes - LONGITUDINE).argmin())
print(f"punto di griglia: {previsione.latitudes[riga]:.2f} N, "
      f"{previsione.longitudes[colonna]:.2f} E")

import polars as pl

pl.DataFrame({
    "istante": list(previsione.valid_times),
    "temperatura_C": [float(previsione.t2m_mean[s, riga, colonna] - 273.15)
                      for s in range(previsione.n_lead)],
    "incertezza_K": [float(previsione.t2m_std[s, riga, colonna])
                     for s in range(previsione.n_lead)],
    "prob_pioggia": [float(previsione.precip_probability[s, riga, colonna])
                     for s in range(previsione.n_lead)],
    "pioggia_mm": [float(previsione.precip_amount[s, riga, colonna])
                   for s in range(previsione.n_lead)],
    "prob_neve": [float(previsione.snow_probability[s, riga, colonna])
                  for s in range(previsione.n_lead)],
})

## 6. Verifica contro l'osservato

Poiche' la previsione riguarda giorni gia' trascorsi, l'osservato puo' esistere: quando
c'e', si misura l'errore davvero commesso invece di limitarsi a guardare le mappe.

Attenzione pero': la previsione qui sopra parte dalla finestra **piu' recente**, e per
costruzione i suoi tre giorni previsti cadono oltre l'ultimo slot ingerito. Per
verificare serve una finestra piu' arretrata, che abbia anche i target.

In [ ]:
n_input, n_output = config.windows.input_slots, config.windows.output_slots

def finestra_verificabile(utilizzabili):
    """Ultima finestra che ha sia gli input sia i target: solo li' la verifica ha senso."""
    for candidato in range(len(utilizzabili) - n_input - n_output, -1, -1):
        if utilizzabili[candidato : candidato + n_input + n_output].all():
            return candidato
    return None

inizio_verifica = finestra_verificabile(utilizzabili)
if inizio_verifica is None:
    print("Nessuna finestra ha insieme input e osservato: ingerire altri mesi.")
elif inizio_verifica == inizio:
    verifica = previsione
    print("La previsione piu' recente e' gia' verificabile.")
else:
    verifica = predict_window(config, rete, input_layout, output_layout, stats,
                              lettore, inizio_verifica)
    print(f"Verifica su una finestra arretrata, inizializzata al {verifica.init_time} "
          f"(la piu' recente non ha ancora l'osservato).")

In [ ]:
if inizio_verifica is not None:
    osservato = lettore.read_window(inizio_verifica + n_input, n_output)
    errore = verifica.t2m_mean - osservato["t2m"]
    riepilogo = pl.DataFrame({
        "scadenza": list(range(verifica.n_lead)),
        "istante": list(verifica.valid_times),
        "errore_medio_K": [float(errore[s].mean()) for s in range(verifica.n_lead)],
        "rmse_K": [float(np.sqrt((errore[s] ** 2).mean())) for s in range(verifica.n_lead)],
        "incertezza_dichiarata_K": [float(verifica.t2m_std[s].mean())
                                    for s in range(verifica.n_lead)],
    })
else:
    riepilogo = None
riepilogo

Se l'incertezza dichiarata e' molto piu' piccola dell'RMSE effettivo, il modello e'
troppo sicuro di se'; se e' molto piu' grande, e' troppo prudente. Le due colonne
dovrebbero avvicinarsi con l'addestramento.

## 7. Salvare la previsione

In [ ]:
from dwf.predict import forecast_to_table
from dwf.tables import FORECAST, write_table

tabella = forecast_to_table(previsione, stride=4)
destinazione = config.artifacts_dir / f"fold_{FOLD:02d}"
destinazione.mkdir(parents=True, exist_ok=True)
print(write_table(tabella, FORECAST, destinazione), f"({tabella.height:,} righe)")